In [ ]:
!pip install altair --quiet

In [ ]:
import pandas as pd
import altair as alt
from google.colab import files

print('✅ Librerías cargadas correctamente')
print(f'   Altair versión: {alt.__version__}')

✅ Librerías cargadas correctamente
   Altair versión: 5.5.0


In [17]:
uploaded = files.upload()

Saving dt_cambios_temporadas_2018_2025.csv to dt_cambios_temporadas_2018_2025 (3).csv


In [18]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename, encoding='latin-1', sep=';')

print(f'✅ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'   Temporadas disponibles: {sorted(df["Temporada"].unique())}')
df.head()

✅ Dataset cargado: 131 filas × 10 columnas
   Temporadas disponibles: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,Temporada,Equipo,Técnico 1,Técnico 2,Técnico 3,Técnico 4,Técnico 5,Cantidad de técnicos,Cambios de técnico,Posición final torneo
0,2018,CSD Colo-Colo,Héctor Tapia,Pablo Guede,Agustín Salvatierra,NaN,NaN,3,2,5
1,2018,CD Universidad Católica,Beñat San José,NaN,NaN,NaN,NaN,1,0,1
2,2018,Universidad de Chile,Esteban Valencia,Frank Drío Kudelka,Ángel Guillermo Hoyos,NaN,NaN,3,2,3
3,2018,Unión La Calera,Francisco Meneghini,Víctor Rivero,NaN,NaN,NaN,2,1,6
4,2018,Universidad de Concepción,Francisco Bozán,NaN,NaN,NaN,NaN,1,0,2


In [19]:
# Filtrar período de análisis
df_filtrado = df[df['Temporada'].between(2019, 2025)].copy()

# Clasificar en 3 grupos
def clasificar(cambios):
    if cambios == 0:
        return '0 cambios'
    elif cambios <= 2:
        return '1–2 cambios'
    else:
        return '3+ cambios'

df_filtrado['Grupo'] = df_filtrado['Cambios de técnico'].apply(clasificar)

# Calcular promedio de posición final y cantidad de equipos por grupo/año
resumen = (
    df_filtrado
    .groupby(['Temporada', 'Grupo'])
    .agg(
        promedio_posicion=('Posición final torneo', 'mean'),
        n_equipos=('Equipo', 'count')
    )
    .reset_index()
)

resumen['promedio_posicion'] = resumen['promedio_posicion'].round(1)
resumen['label_n'] = resumen['n_equipos'].apply(lambda x: f'n={x}')

# Orden explícito de los grupos para que las barras salgan en ese orden
orden_grupos = ['0 cambios', '1–2 cambios', '3+ cambios']
resumen['Grupo'] = pd.Categorical(resumen['Grupo'], categories=orden_grupos, ordered=True)
resumen = resumen.sort_values(['Temporada', 'Grupo'])

print('Vista previa del resumen:')
print(resumen.to_string(index=False))

Vista previa del resumen:
 Temporada       Grupo  promedio_posicion  n_equipos label_n
      2019   0 cambios                6.5          8     n=8
      2019 1–2 cambios               10.5          8     n=8
      2020   0 cambios                6.7          7     n=7
      2020 1–2 cambios               12.1          9     n=9
      2020  3+ cambios                7.5          2     n=2
      2021   0 cambios                6.6          5     n=5
      2021 1–2 cambios                8.7         10    n=10
      2021  3+ cambios               16.5          2     n=2
      2022   0 cambios                5.5          8     n=8
      2022 1–2 cambios               11.5          8     n=8
      2023   0 cambios                5.1          8     n=8
      2023 1–2 cambios               12.0          7     n=7
      2023  3+ cambios               13.0          1     n=1
      2024   0 cambios                7.2          6     n=6
      2024 1–2 cambios                9.2          9     n=

In [ ]:
COLOR_ESTABLE    = '#2C6EAB'  # azul — sin cambios
COLOR_INTERMEDIO = '#E8A838'  # naranja — 1-2 cambios
COLOR_ROTACION   = '#C94040'  # rojo — 3+ cambios
COLOR_FONDO      = '#F7F7F5'
COLOR_GRILLA     = '#E0DDD8'

orden_grupos = ['0 cambios', '1–2 cambios', '3+ cambios']

# Barras principales
barras = (
    alt.Chart(resumen)
    .mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4, opacity=0.92)
    .encode(
        x=alt.X('Temporada:O',
                axis=alt.Axis(labelFontSize=12, titleFontSize=13, labelAngle=0),
                title='Temporada'),
        y=alt.Y('promedio_posicion:Q',
                scale=alt.Scale(domain=[0, 18]),
                axis=alt.Axis(labelFontSize=11, titleFontSize=13,
                              gridColor=COLOR_GRILLA, tickCount=6),
                title='Posición promedio final (menor = mejor)'),
        color=alt.Color('Grupo:N',
                        scale=alt.Scale(
                            domain=orden_grupos,
                            range=[COLOR_ESTABLE, COLOR_INTERMEDIO, COLOR_ROTACION]),
                        legend=alt.Legend(
                            title='Cambios de técnico',
                            titleFontSize=12,
                            labelFontSize=11,
                            orient='top-right',
                            symbolSize=120
                        )),
        xOffset=alt.XOffset('Grupo:N',
                            scale=alt.Scale(domain=orden_grupos)),
        tooltip=[
            alt.Tooltip('Temporada:O', title='Año'),
            alt.Tooltip('Grupo:N', title='Grupo'),
            alt.Tooltip('promedio_posicion:Q', title='Posición promedio', format='.1f'),
            alt.Tooltip('n_equipos:Q', title='N° de equipos')
        ]
    )
)

# Etiquetas de valor sobre cada barra
etiquetas_valor = (
    alt.Chart(resumen)
    .mark_text(dy=-6, fontSize=10, fontWeight='bold')
    .encode(
        x=alt.X('Temporada:O'),
        y=alt.Y('promedio_posicion:Q'),
        xOffset=alt.XOffset('Grupo:N', scale=alt.Scale(domain=orden_grupos)),
        text=alt.Text('promedio_posicion:Q', format='.1f'),
        color=alt.value('#333333')
    )
)

# Etiquetas n= debajo de cada barra
etiquetas_n = (
    alt.Chart(resumen)
    .mark_text(dy=12, fontSize=8, fontStyle='italic')
    .encode(
        x=alt.X('Temporada:O'),
        y=alt.value(0),
        xOffset=alt.XOffset('Grupo:N', scale=alt.Scale(domain=orden_grupos)),
        text=alt.Text('label_n:N'),
        color=alt.value('#999999')
    )
)

# Línea de referencia: media general
media_general = df_filtrado['Posición final torneo'].mean()
linea_ref = (
    alt.Chart(pd.DataFrame({'y': [media_general]}))
    .mark_rule(strokeDash=[6, 4], color='#999999', strokeWidth=1.2)
    .encode(y='y:Q')
)

etiqueta_ref = (
    alt.Chart(pd.DataFrame({
        'y': [media_general],
        'x': ['2025'],
        'label': [f'Media general ({media_general:.1f})']
    }))
    .mark_text(align='right', dx=-6, dy=-7, fontSize=9, color='#999999')
    .encode(x=alt.X('x:O'), y=alt.Y('y:Q'), text='label:N')
)

# Composición final
grafico = (
    (barras + etiquetas_valor + etiquetas_n + linea_ref + etiqueta_ref)
    .properties(
        width=720,
        height=400,
        title=alt.TitleParams(
            text='Estabilidad técnica y posición final — Primera División de Chile',
            subtitle='Promedio de posición final según número de cambios de técnico por temporada (2019–2025)',
            fontSize=16,
            subtitleFontSize=12,
            subtitleColor='#666666',
            anchor='start',
            offset=12
        ),
        background=COLOR_FONDO
    )
    .configure_view(strokeWidth=0)
    .configure_axis(domainColor=COLOR_GRILLA)
)

grafico

In [ ]:
grafico.save('estabilidad_tecnicos.html')
files.download('estabilidad_tecnicos.html')
print('✅ Gráfico guardado como estabilidad_tecnicos.html')